## Simple Decision Tree model
Build a Decision Tree classifier for the Iris dataset using features served from a Feast feature store, implement hyperparameter tuning with Hyperopt, and track all experiments using MLflow.

### Model Training and Hyperparameter Tuning

In [17]:
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")
%env PYTHONWARNINGS=ignore
%env JUPYTER_PLATFORM_DIRS=1

env: PYTHONWARNINGS=ignore
env: JUPYTER_PLATFORM_DIRS=1


In [18]:
# Import the necessary libraries

import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll.base import scope
import mlflow
from mlflow.models import infer_signature

from feast import FeatureStore

import joblib

# Setting the warnings to be ignore once again after all the importing
import logging
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

In [19]:
# Set MLflow tracking URI
mlflow.set_tracking_uri("http://35.223.195.3:5000/")

In [20]:
# Initialize the Feast feature store
# This connects to the pre-configured feature repository
store = FeatureStore(repo_path="Iris_Feast/feature_repo")

In [21]:
# Load entity dataframe with timestamps for historical feature retrieval
entity_df = pd.read_csv("data/entity.csv", parse_dates=['event_timestamp'])

# Retrieve historical features using Feast
# This gets features at specific points in time for training
data = store.get_historical_features(
    entity_df=entity_df,
    features=store.get_feature_service("feast_model_v1")
).to_df()

# Display first 10 rows of the dataset
print("First 10 rows of the dataset:")
data.head(10)

First 10 rows of the dataset:


,species,event_timestamp,sepal_length,sepal_width,petal_length,petal_width
0,setosa,2025-06-21 21:37:06.043098+00:00,5.2,3.5,1.5,0.2
1,setosa,2025-06-21 23:02:06.043098+00:00,5.1,3.8,1.9,0.4
2,setosa,2025-06-21 20:02:06.043098+00:00,4.4,2.9,1.4,0.2
3,setosa,2025-06-21 20:12:06.043098+00:00,5.4,3.7,1.5,0.2
4,setosa,2025-06-21 21:17:06.043098+00:00,5.1,3.3,1.7,0.5
5,setosa,2025-06-21 20:17:06.043098+00:00,4.8,3.4,1.6,0.2
6,setosa,2025-06-21 21:47:06.043098+00:00,4.7,3.2,1.6,0.2
7,setosa,2025-06-21 20:07:06.043098+00:00,4.9,3.1,1.5,0.1
8,setosa,2025-06-21 20:57:06.043098+00:00,5.1,3.8,1.5,0.3
9,setosa,2025-06-21 19:47:06.043098+00:00,5.4,3.9,1.7,0.4


In [22]:
# Get information about the dataset structure
print("Dataset Information:")
data.info()

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147 entries, 0 to 146
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   species          147 non-null    object             
 1   event_timestamp  147 non-null    datetime64[us, UTC]
 2   sepal_length     147 non-null    float64            
 3   sepal_width      147 non-null    float64            
 4   petal_length     147 non-null    float64            
 5   petal_width      147 non-null    float64            
dtypes: datetime64[us, UTC](1), float64(4), object(1)
memory usage: 7.0+ KB


In [23]:
# Split data into training and testing sets
# Using stratified split to maintain class distribution
random_state = 42
test_size = 0.4 # 40% of the data for testing
train, test = train_test_split(
    data, 
    test_size=test_size, 
    stratify=data['species'],  # Maintain class proportions
    random_state=random_state  # For reproducible results
)

# Prepare feature matrices and target vectors
# Features: sepal and petal measurements
X_train = train[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y_train = train.species

X_test = test[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y_test = test.species

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 88
Test set size: 59


In [24]:
# Create model signature for MLflow
signature = infer_signature(X_train.iloc[:2], y_train[:2])

In [25]:
# Define model creation and training function
def create_and_train_model(max_depth, min_samples_split, min_samples_leaf):
    # Initialize Decision Tree classifier with hyperparameters
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=random_state
    )
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions on test set
    y_pred_test = model.predict(X_test)
    
    # Calculate metrics
    accuracy = metrics.accuracy_score(y_test, y_pred_test)
    precision = metrics.precision_score(y_test, y_pred_test, average='weighted')
    recall = metrics.recall_score(y_test, y_pred_test, average='weighted')
    f1 = metrics.f1_score(y_test, y_pred_test, average='weighted')
    
    return {
        "model": model,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "predictions": y_pred_test
    }

In [26]:
# Define objective function for hyperparameter optimization
def objective(params):
    with mlflow.start_run(nested=True):
        # Log hyperparameters being tested
        mlflow.log_params({
            "max_depth": params["max_depth"],
            "min_samples_split": params["min_samples_split"],
            "min_samples_leaf": params["min_samples_leaf"],
            "random_state": random_state
        })
        
        # Train model with current hyperparameters
        result = create_and_train_model(
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
        )
        
        # Log training results
        mlflow.log_metrics({
            "test_accuracy": result["accuracy"],
            "test_precision": result["precision"],
            "test_recall": result["recall"],
            "test_f1_score": result["f1_score"]
        })
        
        # Log the trained model
        mlflow.sklearn.log_model(
            result["model"], 
            name="decision_tree_model", 
            signature=signature
        )
        
        # Create and log confusion matrix    
        cm = metrics.confusion_matrix(y_test, result["predictions"])
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=pd.unique(y_test), 
                   yticklabels=pd.unique(y_test))
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.tight_layout()
        
        mlflow.log_figure(plt.gcf(), "confusion_matrix.png")
        plt.close()
        
        # Return negative accuracy for Hyperopt (it minimizes)
        return {"loss": -result["accuracy"], "status": STATUS_OK}

In [27]:
# Define search space for hyperparameters
search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 1, 10, 1)),
    "min_samples_split": scope.int(hp.quniform("min_samples_split", 2, 10, 1)),
    "min_samples_leaf": scope.int(hp.quniform("min_samples_leaf", 1, 10, 1)),
}

In [28]:
# Set the name of the experiment
mlflow.set_experiment("iris-classification-optimization")

<Experiment: artifact_location='gs://practice-oppe-arcane-rigging-461217-m1/mlflow/1', creation_time=1752941731336, experiment_id='1', last_update_time=1752941731336, lifecycle_stage='active', name='iris-classification-optimization', tags={}>

In [29]:
with mlflow.start_run(run_name="iris-hyperparameter-sweep"):
    # Log experiment metadata
    max_evals = 10
    mlflow.set_tags({
        "max_evaluations": max_evals,
        "objective_metric": "test_accuracy",
        "dataset": "Iris",
        "model_type": "Decision Tree",
        "feature_store": "Feast",
        "test_size" : f"{test_size*100}%"
    })
    
    # Run optimization
    trials = Trials()
    best_params = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        verbose=True,
    )
    
    # Find and log best results
    best_trial = min(trials.results, key=lambda x: x["loss"])
    best_accuracy = -best_trial["loss"]  # Convert back to positive accuracy
    
    best_max_depth = best_params["max_depth"]
    best_min_samples_split = best_params["min_samples_split"]
    best_min_samples_leaf = best_params["min_samples_leaf"]
    
    # Log optimization results
    mlflow.log_params({
        "best_max_depth": best_max_depth,
        "best_min_samples_split": best_min_samples_split,
        "best_min_samples_leaf": best_min_samples_leaf,
    })
    
    mlflow.log_metrics({
        "best_test_accuracy": best_accuracy,
        "total_trials": len(trials.trials)
    })
    
    print(f"\nOptimization completed!")
    print(f"Best Test accuracy: {best_accuracy:.4f}")
    print(f"Best hyperparameters:")
    print(f"  - Max depth: {best_max_depth}")
    print(f"  - Min samples split: {best_min_samples_split}")
    print(f"  - Min samples leaf: {best_min_samples_leaf}")

🏃 View run melodic-pug-539 at: http://35.223.195.3:5000/#/experiments/1/runs/f88fef3a4d9949059cc976a3042f0790

🧪 View experiment at: http://35.223.195.3:5000/#/experiments/1

🏃 View run vaunted-foal-936 at: http://35.223.195.3:5000/#/experiments/1/runs/a1565f52928b41bbbda605aeb0bf1def

🧪 View experiment at: http://35.223.195.3:5000/#/experiments/1                  

🏃 View run enthused-stag-960 at: http://35.223.195.3:5000/#/experiments/1/runs/38888fdeaaf54470a557373280da349e

🧪 View experiment at: http://35.223.195.3:5000/#/experiments/1                  

🏃 View run bemused-conch-461 at: http://35.223.195.3:5000/#/experiments/1/runs/46f346a25d504b76a4782a37c607c23e

🧪 View experiment at: http://35.223.195.3:5000/#/experiments/1                  

🏃 View run puzzled-trout-419 at: http://35.223.195.3:5000/#/experiments/1/runs/ae510208a0d04324828b78acd0906c10

🧪 View experiment at: http://35.223.195.3:5000/#/experiments/1                  

🏃 View run classy-crab-252 at: http://35.223.1

In [30]:
# Train final model with best parameters
final_model = DecisionTreeClassifier(
    max_depth=int(best_max_depth),
    min_samples_split=int(best_min_samples_split),
    min_samples_leaf=int(best_min_samples_leaf),
    random_state=random_state
)

final_model.fit(X_train, y_train)

# Make predictions on test set
y_pred_test = final_model.predict(X_test)

# Calculate and display accuracy
accuracy = metrics.accuracy_score(y_test, y_pred_test)
print(f'The accuracy of the Decision Tree is {accuracy:.3f}')

The accuracy of the Decision Tree is 0.966


In [31]:
# Log final model
with mlflow.start_run(run_name="final-model"):
    mlflow.log_params({
        "max_depth": best_max_depth,
        "min_samples_split": best_min_samples_split,
        "min_samples_leaf": best_min_samples_leaf,
        "random_state":random_state,
        "model_type": "Decision Tree"
    })
    
    mlflow.log_metric("test_accuracy",accuracy)
    
    mlflow.sklearn.log_model(final_model, "final_model", signature=signature)


🏃 View run final-model at: http://35.223.195.3:5000/#/experiments/1/runs/f5ee39bc6ae54795a5ac8b17cd862192
🧪 View experiment at: http://35.223.195.3:5000/#/experiments/1


### Save the model

In [24]:
# Save the trained model for future use
joblib.dump(final_model, "artifacts/model.joblib")
print("Model saved successfully to artifacts/model.joblib")

Model saved successfully to artifacts/model.joblib


### Online inferencing using the trained model

In [32]:
# Load the trained model
model = joblib.load("artifacts/model.joblib")

In [33]:
# Demonstrate online feature serving
# Get the latest features for each species
features = store.get_online_features(
    features=store.get_feature_service("feast_model_v1"),
    entity_rows=[
        {"species": "setosa"},
        {"species": "versicolor"}, 
        {"species": "virginica"}
    ],
).to_df()

print("Online features retrieved:")
features

Online features retrieved:


,species,petal_length,sepal_length,sepal_width,petal_width
0,setosa,1.4,5.0,3.3,0.2
1,versicolor,4.1,5.7,2.8,1.3
2,virginica,5.1,5.9,3.0,1.8


In [34]:
# Apply trained model to online features
features["model_prediction"] = model.predict(
    features[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
)

print("Features with model predictions:")
features

Features with model predictions:


,species,petal_length,sepal_length,sepal_width,petal_width,model_prediction
0,setosa,1.4,5.0,3.3,0.2,setosa
1,versicolor,4.1,5.7,2.8,1.3,versicolor
2,virginica,5.1,5.9,3.0,1.8,virginica
